In [ ]:
import torch
import torch.nn as nn

class BaselineCNN(nn.Module):
    def __init__(self, in_channels: int = 3, base_channels: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, in_channels, kernel_size=3, padding=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class ResidualCNN(nn.Module):
    def __init__(self, in_channels: int = 3, base_channels: int = 64):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(base_channels, base_channels, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(base_channels, in_channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.conv3(x)
        return residual + x

In [ ]:
import argparse
from pathlib import Path
import sys

root_dir = Path.cwd()  # In Colab, use current working directory
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import torch
from PIL import Image
import torchvision.transforms as T

# Models are defined in the previous cell

ModuleNotFoundError: No module named 'models'

In [ ]:
def load_image(path: str, image_size: int = 128):
    img = Image.open(path).convert("RGB")
    transform = T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
    ])
    return transform(img).unsqueeze(0)  # [1, C, H, W]


def save_image(tensor: torch.Tensor, path: str):
    tensor = tensor.clamp(0, 1).squeeze(0)
    img = T.ToPILImage()(tensor.cpu())
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    img.save(path)


def get_model(name: str, checkpoint: str = None, device: str = "cpu"):
    if name == "residual":
        model = ResidualCNN()
    else:
        model = BaselineCNN()
    model.to(device)

    if checkpoint is not None and Path(checkpoint).exists():
        state = torch.load(checkpoint, map_location=device)
        if "model_state" in state:
            model.load_state_dict(state["model_state"])
        else:
            model.load_state_dict(state)
        print(f"Loaded weights from {checkpoint}")
    else:
        print("Warning: No checkpoint loaded, using random weights.")

    model.eval()
    return model

In [ ]:
def main(args):
    print(f"Input image: {args.input}")
    print(f"Output path: {args.output}")
    print(f"Model: {args.model}")
    print(f"Image size: {args.image_size}")
    if args.checkpoint:
        print(f"Checkpoint: {args.checkpoint}")
    else:
        print("No checkpoint provided, using random weights")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    model = get_model(args.model, args.checkpoint, device)

    noisy = load_image(args.input, args.image_size).to(device)
    print(f"Loaded noisy image shape: {noisy.shape}")

    with torch.no_grad():
        denoised = model(noisy)
    print(f"Denoised image shape: {denoised.shape}")

    save_image(denoised, args.output)
    print(f"Denoised image saved to {args.output}")

In [ ]:
# For Colab, use simple argument parsing or hardcode values
import argparse

# Example usage in Colab:
# Set your input and output paths
input_path = "/content/noisy_image.png"  # Upload your image to Colab
output_path = "/content/denoised_image.png"
model_name = "residual"  # or "baseline"
checkpoint_path = None  # or "/content/model.pth"
image_size = 128

# Create args object
args = argparse.Namespace(
    input=input_path,
    output=output_path,
    model=model_name,
    checkpoint=checkpoint_path,
    image_size=image_size
)

main(args)